# Phase 4 — within-block vigor zones

Per-pixel seasonal NDVI profiles (monthly medians Apr–Oct via `toBands`),
one `sampleRegions` call per block-season, then **k-means (k=3) locally** to
split each block-season into low / medium / high vigor zones.

The only Earth Engine work is the per-pixel sampling in `vigor.extract`
(`sample_seasonal_profiles` → `fetch_pixels`); clustering and mapping are
pure sklearn/matplotlib in `vigor.zones` / `vigor.plots`, offline. The pulled
pixel table is cached to `data/raw/` so re-running the analysis needs no EE.

**Acceptance criteria (Phase 4):**
- Zones that stay stable across years read as soil or rootstock.
- Zones that appear in a single year read as management, irrigation, or damage.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from vigor import extract, zones, plots

PIXELS_CSV = PROJECT_ROOT / "data" / "raw" / "pixel_profiles.csv"
ZONES_PARQUET = PROJECT_ROOT / "data" / "processed" / "vigor_zones.parquet"
FIGURES = PROJECT_ROOT / "outputs" / "figures"

# Complete growing seasons only (2026 is still in progress).
YEARS = list(range(2019, 2026))
K = 3

In [ ]:
extract.init_ee()
blocks = extract.load_blocks(PROJECT_ROOT / "data" / "blocks.geojson")
blocks[["block_id", "site", "variety", "planting_year", "n_pixels"]]

## Per-pixel sampling

One `getInfo` per season (each is only a few hundred pixels × 7 bands, well
within limits), concatenated and cached. Delete `data/raw/pixel_profiles.csv`
to force a fresh pull after changing blocks or the season window.

In [ ]:
if PIXELS_CSV.exists():
    pixels = pd.read_csv(PIXELS_CSV)
    print(f"using cached pixel table: {len(pixels)} rows")
else:
    frames = []
    for year in YEARS:
        px = extract.fetch_pixels(extract.sample_seasonal_profiles(blocks, year))
        print(f"{year}: {len(px)} pixels")
        frames.append(px)
    pixels = pd.concat(frames, ignore_index=True)
    PIXELS_CSV.parent.mkdir(parents=True, exist_ok=True)
    pixels.to_csv(PIXELS_CSV, index=False)
    print(f"\ncached -> {PIXELS_CSV}")

pixels.groupby(["block_id", "year"]).size().unstack()

## Cluster into zones (offline)

k-means per block-season; labels reordered so zone 0 is always lowest vigor
and zone k−1 highest, making "the high-vigor zone" mean the same thing across
years.

In [ ]:
zoned = zones.zone_all(pixels, k=K)
zones.write_zones(zoned, ZONES_PARQUET)
print(f"zones -> {ZONES_PARQUET}\n")
zones.zone_summary(zoned).pivot(index=["block_id", "zone"], columns="year", values="n_pixels")

## Zone maps

One panel per season per block; darker = higher vigor. Look for zones that
hold their spatial shape year to year (soil/rootstock) versus zones that show
up in a single season (management, irrigation, or damage).

In [ ]:
for block_id in sorted(zoned["block_id"].unique()):
    fig = plots.zone_maps(zoned, block_id, k=K)
    saved = plots.save_figure(fig, FIGURES / f"04_{block_id}_vigor_zones.png")
    print(f"figure -> {saved}")

## Zone mean NDVI by year

Mean seasonal NDVI of each zone, per block per year — a numeric read of how
separated the zones are and how each block's vigor structure shifts over time.

In [ ]:
zones.zone_summary(zoned).pivot(index=["block_id", "zone"], columns="year", values="mean_ndvi").round(3)